# **YOLO11 Pytorch Implementation :**

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.append(
    str(PROJECT_ROOT)
)

## ***blocks :***

In [2]:
from src.models.blocks import ConvBlock

In [3]:
import torch

x = torch.randn(
    1,
    3,
    640,
    640
)

print(x.shape)

torch.Size([1, 3, 640, 640])


In [4]:
conv = ConvBlock(
    in_channels=3,
    out_channels=16,
    kernel_size=3,
    stride=2
)

In [5]:
y = conv(x)

print(y.shape)

torch.Size([1, 16, 320, 320])


### ***Bottleneck :***

In [6]:
from src.models.blocks import Bottleneck

In [7]:
x = torch.randn(
    1,
    64,
    80,
    80
)

In [8]:
bottleneck = Bottleneck(
    in_channels=64,
    out_channels=64
)

In [9]:
y = bottleneck(x)

print("Input :", x.shape)
print("Output:", y.shape)

Input : torch.Size([1, 64, 80, 80])
Output: torch.Size([1, 64, 80, 80])


In [10]:
# # * with shortcut
bottleneck = Bottleneck(
    in_channels=64,
    out_channels=128,
    shortcut=True
)

y = bottleneck(x)

print("Input :", x.shape)
print("Output:", y.shape)

Input : torch.Size([1, 64, 80, 80])
Output: torch.Size([1, 128, 80, 80])


### ***C3k***

In [11]:
from src.models.blocks import C3k

In [12]:
x = torch.randn(
    1,
    64,
    80,
    80
)

In [13]:
c3k = C3k(
    in_channels=64,
    out_channels=128,
    num_bottlenecks=2
)

In [14]:
y = c3k(x)

print("Input :", x.shape)
print("Output:", y.shape)

Input : torch.Size([1, 64, 80, 80])
Output: torch.Size([1, 128, 80, 80])


### ***C3k2***

In [15]:
from src.models.blocks import C3k2

In [16]:
x = torch.randn(
    1,
    64,
    80,
    80
)

In [17]:
c3k2 = C3k2(
    in_channels=64,
    out_channels=128,
    num_blocks=2
)

In [18]:
y = c3k2(x)

print("Input :", x.shape)
print("Output:", y.shape)

Input : torch.Size([1, 64, 80, 80])
Output: torch.Size([1, 128, 80, 80])


In [19]:
def count_parameters(model):

    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

In [20]:
print(
    "ConvBlock parameters:",
    count_parameters(conv)
)

print(
    "Bottleneck parameters:",
    count_parameters(bottleneck)
)

print(
    "C3k parameters:",
    count_parameters(c3k)
)

print(
    "C3k2 parameters:",
    count_parameters(c3k2)
)

ConvBlock parameters: 464
Bottleneck parameters: 110976
C3k parameters: 173056
C3k2 parameters: 206592


# **SPPF**

In [21]:
from src.models.attention import SPPF

In [22]:
x = torch.randn(
    1,
    256,
    40,
    40
)

In [23]:
sppf = SPPF(
    in_channels=256,
    out_channels=256
)

In [24]:
y = sppf(x)

print("Input :", x.shape)
print("Output:", y.shape)

Input : torch.Size([1, 256, 40, 40])
Output: torch.Size([1, 256, 40, 40])


# **PSABlock**

In [25]:
from src.models.attention import PSABlock

In [26]:
x = torch.randn(
    2,
    128,
    20,
    20
)

In [27]:
psa = PSABlock(
    channels=128,
    num_heads=4
)

In [28]:
y = psa(x)

print("Input :", x.shape)
print("Output:", y.shape)

Input : torch.Size([2, 128, 20, 20])
Output: torch.Size([2, 128, 20, 20])


# **C2PSA**

In [29]:
from src.models.attention import C2PSA

In [30]:
x = torch.randn(
    2,
    256,
    20,
    20
)

In [31]:
c2psa = C2PSA(
    in_channels=256,
    out_channels=256,
    num_blocks=1,
    num_heads=4
)

In [32]:
y = c2psa(x)

print("Input :", x.shape)
print("Output:", y.shape)

Input : torch.Size([2, 256, 20, 20])
Output: torch.Size([2, 256, 20, 20])


In [33]:
x = torch.randn(
    1,
    256,
    20,
    20
)


sppf = SPPF(
    in_channels=256,
    out_channels=256
)

c2psa = C2PSA(
    in_channels=256,
    out_channels=256,
    num_blocks=1,
    num_heads=4
)

x_sppf = sppf(x)

x_c2psa = c2psa(
    x_sppf
)

print(
    "Input:",
    x.shape
)

print(
    "After SPPF:",
    x_sppf.shape
)

print(
    "After C2PSA:",
    x_c2psa.shape
)

Input: torch.Size([1, 256, 20, 20])
After SPPF: torch.Size([1, 256, 20, 20])
After C2PSA: torch.Size([1, 256, 20, 20])


In [34]:
print(
    "SPPF parameters:",
    count_parameters(sppf)
)

print(
    "PSABlock parameters:",
    count_parameters(psa)
)

print(
    "C2PSA parameters:",
    count_parameters(c2psa)
)


SPPF parameters: 164608
PSABlock parameters: 82944
C2PSA parameters: 215040


# ***Backbone :***

In [35]:
import torch

from src.models.backbone import YOLO11Backbone

In [36]:
x = torch.randn(
    1,
    3,
    640,
    640
)

print(
    "Input:",
    x.shape
)

Input: torch.Size([1, 3, 640, 640])


In [37]:
backbone = YOLO11Backbone()

print(backbone)

YOLO11Backbone(
  (stem): ConvBlock(
    (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (act): SiLU(inplace=True)
  )
  (down1): ConvBlock(
    (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (act): SiLU(inplace=True)
  )
  (c3k2_1): C3k2(
    (cv1): ConvBlock(
      (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (blocks): ModuleList(
      (0): C3k(
        (branch1): ConvBlock(
          (conv): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (act): SiL

In [38]:
p3, p4, p5 = backbone(x)

In [39]:
print("P3:", p3.shape)
print("P4:", p4.shape)
print("P5:", p5.shape)

P3: torch.Size([1, 64, 80, 80])
P4: torch.Size([1, 128, 40, 40])
P5: torch.Size([1, 256, 20, 20])


In [40]:
# # * Verify strides

input_size = 640

print(
    "P3 stride:",
    input_size // p3.shape[-1]
)

print(
    "P4 stride:",
    input_size // p4.shape[-1]
)

print(
    "P5 stride:",
    input_size // p5.shape[-1]
)

P3 stride: 8
P4 stride: 16
P5 stride: 32


In [41]:
total_params = sum(
    p.numel()
    for p in backbone.parameters()
)

trainable_params = sum(
    p.numel()
    for p in backbone.parameters()
    if p.requires_grad
)

print(
    f"Total parameters: "
    f"{total_params:,}"
)

print(
    f"Trainable parameters: "
    f"{trainable_params:,}"
)

Total parameters: 1,905,200
Trainable parameters: 1,905,200


In [42]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

Device: cuda


In [43]:
backbone = backbone.to(device)

x = x.to(device)

with torch.no_grad():

    p3, p4, p5 = backbone(x)

print(p3.shape)
print(p4.shape)
print(p5.shape)

torch.Size([1, 64, 80, 80])
torch.Size([1, 128, 40, 40])
torch.Size([1, 256, 20, 20])


# **FPN :**

In [44]:
import torch

from src.models.neck import YOLO11Neck

In [45]:
p3 = torch.randn(
    1,
    64,
    80,
    80
)

p4 = torch.randn(
    1,
    128,
    40,
    40
)

p5 = torch.randn(
    1,
    256,
    20,
    20
)


print("P3:", p3.shape)
print("P4:", p4.shape)
print("P5:", p5.shape)

P3: torch.Size([1, 64, 80, 80])
P4: torch.Size([1, 128, 40, 40])
P5: torch.Size([1, 256, 20, 20])


In [46]:
neck = YOLO11Neck()

print(neck)

YOLO11Neck(
  (reduce_p5): ConvBlock(
    (conv): Conv2d(256, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (act): SiLU(inplace=True)
  )
  (fpn_p4): C3k2(
    (cv1): ConvBlock(
      (conv): Conv2d(256, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (blocks): ModuleList(
      (0): C3k(
        (branch1): ConvBlock(
          (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (branch2): ConvBlock(
          (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (a

In [47]:
f3, f4, f5 = neck(
    p3,
    p4,
    p5
)

In [48]:
print("F3:", f3.shape)
print("F4:", f4.shape)
print("F5:", f5.shape)

F3: torch.Size([1, 64, 80, 80])
F4: torch.Size([1, 128, 40, 40])
F5: torch.Size([1, 256, 20, 20])


In [49]:
input_size = 640

print(
    "F3 stride:",
    input_size // f3.shape[-1]
)

print(
    "F4 stride:",
    input_size // f4.shape[-1]
)

print(
    "F5 stride:",
    input_size // f5.shape[-1]
)

F3 stride: 8
F4 stride: 16
F5 stride: 32


In [50]:
# # * batch = 2

p3 = torch.randn(
    2,
    64,
    80,
    80
)

p4 = torch.randn(
    2,
    128,
    40,
    40
)

p5 = torch.randn(
    2,
    256,
    20,
    20
)

f3, f4, f5 = neck(
    p3,
    p4,
    p5
)

print(f3.shape)
print(f4.shape)
print(f5.shape)

torch.Size([2, 64, 80, 80])
torch.Size([2, 128, 40, 40])
torch.Size([2, 256, 20, 20])


In [51]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

Device: cuda


In [52]:
neck = neck.to(device)

p3 = p3.to(device)
p4 = p4.to(device)
p5 = p5.to(device)

In [53]:
with torch.no_grad():

    f3, f4, f5 = neck(
        p3,
        p4,
        p5
    )

print(f3.shape)
print(f4.shape)
print(f5.shape)

torch.Size([2, 64, 80, 80])
torch.Size([2, 128, 40, 40])
torch.Size([2, 256, 20, 20])


In [54]:
total_params = sum(
    p.numel()
    for p in neck.parameters()
)

trainable_params = sum(
    p.numel()
    for p in neck.parameters()
    if p.requires_grad
)

print(
    f"Total parameters: "
    f"{total_params:,}"
)

print(
    f"Trainable parameters: "
    f"{trainable_params:,}"
)

Total parameters: 1,027,904
Trainable parameters: 1,027,904


# **Head :**

In [55]:
import torch

from src.models.head import DetectionHead

In [56]:
NUM_CLASSES = 10

In [57]:
p3 = torch.randn(
    2,
    64,
    80,
    80
)

In [58]:
head = DetectionHead(
    in_channels=64,
    num_classes=NUM_CLASSES,
    reg_max=16
)

In [59]:
output = head(p3)

In [60]:
print(
    "Box:",
    output["box"].shape
)

print(
    "Class:",
    output["cls"].shape
)

Box: torch.Size([2, 64, 80, 80])
Class: torch.Size([2, 10, 80, 80])


In [61]:
p4 = torch.randn(
    2,
    128,
    40,
    40
)

head_p4 = DetectionHead(
    in_channels=128,
    num_classes=NUM_CLASSES,
    reg_max=16
)

output_p4 = head_p4(p4)

print(
    "Box:",
    output_p4["box"].shape
)

print(
    "Class:",
    output_p4["cls"].shape
)

Box: torch.Size([2, 64, 40, 40])
Class: torch.Size([2, 10, 40, 40])


In [62]:
p5 = torch.randn(
    2,
    256,
    20,
    20
)

head_p5 = DetectionHead(
    in_channels=256,
    num_classes=NUM_CLASSES,
    reg_max=16
)

output_p5 = head_p5(p5)

print(
    "Box:",
    output_p5["box"].shape
)

print(
    "Class:",
    output_p5["cls"].shape
)

Box: torch.Size([2, 64, 20, 20])
Class: torch.Size([2, 10, 20, 20])


# **multi-scale head**

In [63]:
from src.models.head import YOLO11Detect

In [64]:
f3 = torch.randn(
    2,
    64,
    80,
    80
)

f4 = torch.randn(
    2,
    128,
    40,
    40
)

f5 = torch.randn(
    2,
    256,
    20,
    20
)

In [65]:
detect = YOLO11Detect(
    num_classes=10,
    reg_max=16
)

In [66]:
outputs = detect(
    f3,
    f4,
    f5
)

In [67]:
for level, output in outputs.items():

    print(
        f"\n{level}"
    )

    print(
        "Box:",
        output["box"].shape
    )

    print(
        "Class:",
        output["cls"].shape
    )


p3
Box: torch.Size([2, 64, 80, 80])
Class: torch.Size([2, 10, 80, 80])

p4
Box: torch.Size([2, 64, 40, 40])
Class: torch.Size([2, 10, 40, 40])

p5
Box: torch.Size([2, 64, 20, 20])
Class: torch.Size([2, 10, 20, 20])
